**The idea is to use MacheinLearning and Deeplearning etc. methord to get the First Class models. And using the SVM to do a voting for prediction (Second Class model).**

Some detail information:
We would use **3 different metord to contribute the SVM**.
1. XGBoost (ML)
2. LightGBM (ML)
3. ChemBERTa (DL)

# Import library

In [1]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
from rdkit import RDLogger
import os
import json
import pandas as pd
from pathlib import Path
import multiprocessing as mp
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, rdmolops
from typing import List, Tuple
from sklearn.model_selection import train_test_split, KFold
import xgboost as xgb
import gc, lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from collections import defaultdict
from sklearn.multioutput import MultiOutputRegressor
import networkx as nx
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')

Processing /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl


# Data Processing

## RANDOM SEED

In [2]:
SEED = 42
np.random.seed(SEED)

## Load data

In [3]:
# 1) Load the original train & test
DATA_DIR = Path("/kaggle/input/neurips-open-polymer-prediction-2025")
train_df    = pd.read_csv(DATA_DIR / "train.csv") 
test_df     = pd.read_csv(DATA_DIR / "test.csv")

## External dataset

In [4]:
# -----------------------------------------------------------------------------------
# 2) Canonicalise SMILES to avoid duplicates like '*C=C(*)C' vs '*C(=C*)C'
# -----------------------------------------------------------------------------------
def make_smile_canonical(smile: str):
    try:
        mol = Chem.MolFromSmiles(smile)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return np.nan

train_df["SMILES"] = train_df["SMILES"].apply(make_smile_canonical)
test_df["SMILES"]  = test_df["SMILES"].apply(make_smile_canonical)

# Keep a frozen copy of the mother dataset (for bias correction only)
base_train_df = train_df.copy()

# -----------------------------------------------------------------------------------
# 3) Load external public datasets
# -----------------------------------------------------------------------------------
# 3-1) Tc dataset
data_tc = (
    pd.read_csv("/kaggle/input/tc-smiles/Tc_SMILES.csv")
      .rename(columns={"TC_mean": "Tc"})
)

# 3-2) Tg supplement 1
data_tg2 = (
    pd.read_csv("/kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv",
                usecols=["SMILES", "Tg (C)"])
      .rename(columns={"Tg (C)": "Tg"})
)

# 3-3) Tg supplement 2
data_tg3 = (
    pd.read_excel("/kaggle/input/smiles-extra-data/data_tg3.xlsx")
      .rename(columns={"Tg [K]": "Tg"})
)
data_tg3["Tg"] = data_tg3["Tg"] - 273.15          # Kelvin → °C

# 3-4) Density dataset
data_dnst = (
    pd.read_excel("/kaggle/input/smiles-extra-data/data_dnst1.xlsx")
      .rename(columns={"density(g/cm3)": "Density"})[["SMILES", "Density"]]
)
data_dnst["SMILES"] = data_dnst["SMILES"].apply(make_smile_canonical)
# Remove odd entries and apply author’s correction
data_dnst = (
    data_dnst
      .query("SMILES.notnull() and Density.notnull() and Density != 'nylon'")
      .assign(Density=lambda df: df["Density"].astype(float) - 0.118)
)

# 3-5) Tc supplement (NeurIPS dataset 1)
data_tc4 = (
    pd.read_csv("/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv")
      .rename(columns={"TC_mean": "Tc"})
)

# 3-6) Tg supplement (NeurIPS dataset 3)
data_tg4 = (
    pd.read_csv("/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv")
      .rename(columns={"Tg": "Tg"})
)

# 3-7) FFV supplement (NeurIPS dataset 4)
data_ffv = (
    pd.read_csv("/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv")
      .rename(columns={"FFV": "FFV"})
)
data_ffv["FFV"] = pd.to_numeric(data_ffv["FFV"], errors="coerce")

# -----------------------------------------------------------------------------------
# 4) Helper to merge an extra dataset into train_df, with optional bias correction
# -----------------------------------------------------------------------------------
def add_extra_data(df_train: pd.DataFrame,
                   df_extra: pd.DataFrame,
                   target: str,
                   dataset_name: str = "extra",
                   base_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    * Canonicalise SMILES in the extra dataset.
    * If overlap with the mother dataset exists, compute mean difference (extra - base) on overlaps
      and shift the entire external target by that delta (per-target correction). Print the delta.
    * For SMILES already in the competition data, fill **missing** target values only.
    * Fully new SMILES are appended to the training set.
    """
    count_before = df_train[target].notna().sum()

    # Prepare external (canonical & aggregated)
    df_extra = (
        df_extra
          .assign(SMILES=lambda d: d["SMILES"].apply(make_smile_canonical))
          .dropna(subset=["SMILES"])
          .groupby("SMILES", as_index=False)[target].mean()
    )

    # ----- Bias correction against the frozen mother dataset -----
    ref = base_df if base_df is not None else df_train
    base_map = (
        ref[["SMILES", target]]
          .dropna(subset=["SMILES", target])
          .assign(SMILES=lambda d: d["SMILES"].apply(make_smile_canonical))
          .groupby("SMILES", as_index=False)[target].mean()
    )

    overlap = df_extra.merge(base_map, on="SMILES", how="inner", suffixes=("_ext", "_base"))
    n_overlap = len(overlap)

    if n_overlap > 0:
        mean_diff = (overlap[f"{target}_ext"] - overlap[f"{target}_base"]).mean()
        # Shift external data to the mother dataset scale
        df_extra[target] = df_extra[target] - mean_diff
        print(f"[{dataset_name} | {target}] bias correction using {n_overlap} overlaps → Δ={mean_diff:+.6f}")
    else:
        print(f"[{dataset_name} | {target}] no overlap with mother dataset → no correction applied")

    # ----- Merge into current train (fill missing targets for existing SMILES; append new SMILES) -----
    intersect_smiles = set(df_train["SMILES"]) & set(df_extra["SMILES"])
    new_smiles       = set(df_extra["SMILES"]) - set(df_train["SMILES"])

    # Fill missing target values for overlapping SMILES
    if intersect_smiles:
        ext_map = dict(zip(df_extra["SMILES"], df_extra[target]))
        mask_overlap_missing = (df_train["SMILES"].isin(intersect_smiles)) & (df_train[target].isna())
        df_train.loc[mask_overlap_missing, target] = df_train.loc[mask_overlap_missing, "SMILES"].map(ext_map).values

    # Append completely new rows (only SMILES + this target)
    df_train = pd.concat(
        [df_train, df_extra[df_extra["SMILES"].isin(new_smiles)][["SMILES", target]]],
        ignore_index=True,
    )

    added = df_train[target].notna().sum() - count_before
    print(f'[{target}] added {added:>4} usable samples | new SMILES: {len(new_smiles)}')
    return df_train

# -----------------------------------------------------------------------------------
# 5) Merge the external datasets (with per-target bias correction if overlaps exist)
# -----------------------------------------------------------------------------------
train_df = add_extra_data(train_df, data_tc,   "Tc",      dataset_name="TC_public",     base_df=base_train_df)
train_df = add_extra_data(train_df, data_tc4,  "Tc",      dataset_name="TC_supp_ds1",   base_df=base_train_df)
train_df = add_extra_data(train_df, data_tg2,  "Tg",      dataset_name="TG_sup1_JCIM",  base_df=base_train_df)
train_df = add_extra_data(train_df, data_tg3,  "Tg",      dataset_name="TG_sup2_xlsx",  base_df=base_train_df)
train_df = add_extra_data(train_df, data_tg4,  "Tg",      dataset_name="TG_supp_ds3",   base_df=base_train_df)
train_df = add_extra_data(train_df, data_dnst, "Density", dataset_name="Density_sup1",  base_df=base_train_df)
train_df = add_extra_data(train_df, data_ffv,  "FFV",     dataset_name="FFV_supp_ds4",  base_df=base_train_df)

# -----------------------------------------------------------------------------------
# 6) Quick summary
# -----------------------------------------------------------------------------------
print("\n=== Sample counts per target (non-NaN) ===")
for col in ["Tg", "FFV", "Tc", "Density", "Rg"]:
    print(f"{col:>8}: {train_df[col].notna().sum():>6}")

print(f"\ntrain_df shape: {train_df.shape}")
print(f"test_df  shape: {test_df.shape}")

[TC_public | Tc] bias correction using 737 overlaps → Δ=-0.000000
[Tc] added  129 usable samples | new SMILES: 129
[TC_supp_ds1 | Tc] bias correction using 737 overlaps → Δ=-0.000000
[Tc] added    0 usable samples | new SMILES: 0
[TG_sup1_JCIM | Tg] bias correction using 511 overlaps → Δ=+0.000000
[Tg] added  151 usable samples | new SMILES: 136
[TG_sup2_xlsx | Tg] no overlap with mother dataset → no correction applied
[Tg] added  499 usable samples | new SMILES: 499
[TG_supp_ds3 | Tg] no overlap with mother dataset → no correction applied
[Tg] added    0 usable samples | new SMILES: 0
[Density_sup1 | Density] bias correction using 144 overlaps → Δ=+0.000134
[Density] added  634 usable samples | new SMILES: 524
[FFV_supp_ds4 | FFV] no overlap with mother dataset → no correction applied
[FFV] added  862 usable samples | new SMILES: 819

=== Sample counts per target (non-NaN) ===
      Tg:   1161
     FFV:   7892
      Tc:    866
 Density:   1247
      Rg:    614

train_df shape: (10080,

## Features generation

In [5]:
# --------------------- universal hyper-params (both sets share) -------------
DESC_FUNCS = [d[1] for d in Descriptors._descList]     # 2D phys-chem
DESC_NAMES = [d[0] for d in Descriptors._descList]

RADIUS = 2          # Morgan radius
FP_ECFP_1024 = 1024 # for XGBoost
FP_ECFP_2048 = 2048 # for LightGBM
FP_ATOMPAIR   = 2048

N_CPUS = max(mp.cpu_count() - 1, 1)

# --------------------- helper ------------------------------------------------
def mol_from_smiles(sm):
    try:
        return Chem.MolFromSmiles(sm)
    except Exception:
        return None

def bitvect_to_numpy(bv):          # RDKit ExplicitBitVect → np.uint8 array
    return np.frombuffer(bv.ToBitString().encode("utf-8"), "S1").astype(np.uint8)

def featurise(smiles: str, kind: str):
    """
    kind = 'xgb'  -> desc + ECFP1024
    kind = 'lgbm' -> desc + ECFP2048 + AtomPair2048
    """
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None   # will be filtered later

    feat = []

    # ==== continuous descriptors ====
    feat.extend(f(mol) for f in DESC_FUNCS)

    # ==== fingerprints ====
    if kind == "xgb":
        bv = AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=FP_ECFP_1024)
        feat.extend(bitvect_to_numpy(bv))
    else:  # 'lgbm'
        bv1 = AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=FP_ECFP_2048)
        bv2 = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=FP_ATOMPAIR)
        feat.extend(bitvect_to_numpy(bv1))
        feat.extend(bitvect_to_numpy(bv2))

    # ==== gragh feature ====
    adj = rdmolops.GetAdjacencyMatrix(mol)
    G   = nx.from_numpy_array(adj)

    if nx.is_connected(G):
        feat.append(nx.diameter(G))                       # graph_diameter
        feat.append(nx.average_shortest_path_length(G))   # avg_shortest_path
    else:
        feat.extend([0, 0])

    feat.append(len(nx.cycle_basis(G)))                   # num_cycles

    return feat

def build_all_features(df: pd.DataFrame, kind: str):
    pool  = mp.Pool(N_CPUS)
    feats = pool.starmap(featurise, [(sm, kind) for sm in df["SMILES"]])
    pool.close(); pool.join()
    # keep only successful mol→feats
    good = [i for i,f in enumerate(feats) if f is not None]
    X    = np.array([feats[i] for i in good], dtype=np.float32)
    # zero‐fill NaN/Inf
    X[np.isinf(X)] = np.nan
    #X    = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, good

# 1) Featurize TRAIN set
X_all_xgb, idx_train = build_all_features(train_df, kind="xgb")
X_all_lgbm, _        = build_all_features(train_df, kind="lgbm")

# Align Y to X_all_xgb
targets = ["Tg", "FFV", "Tc", "Density", "Rg"]
Y_all = train_df.loc[idx_train, targets].values  # shape = (n_train,5)

# 2) Featurize TEST set
X_test_xgb, idx_test_xgb   = build_all_features(test_df, kind="xgb")
X_test_lgbm, idx_test_lgbm = build_all_features(test_df, kind="lgbm")
test_ids_xgb  = test_df["id"].values[idx_test_xgb]
test_ids_lgbm = test_df["id"].values[idx_test_lgbm]

print(f"Train XGB features:   {X_all_xgb.shape}")
print(f"Train LightGBM feats: {X_all_lgbm.shape}")
print(f"Test  XGB features:   {X_test_xgb.shape}")
print(f"Test  LightGBM feats: {X_test_lgbm.shape}")

Train XGB features:   (10080, 1244)
Train LightGBM feats: (10080, 4316)
Test  XGB features:   (3, 1244)
Test  LightGBM feats: (3, 4316)


## Saving features as file for save time

In [6]:
# ==== save_all_features.py ====
import joblib, os

FILE = "/kaggle/working/polymer_feats.pkl"

joblib.dump(
    {
        "X_all_xgb"     : X_all_xgb,
        "X_all_lgbm"    : X_all_lgbm,
        "idx_train"     : idx_train,
        "Y_all"         : Y_all,
        "X_test_xgb"    : X_test_xgb,
        "X_test_lgbm"   : X_test_lgbm,
        "idx_test_xgb"  : idx_test_xgb,
        "idx_test_lgbm" : idx_test_lgbm,
        "test_ids_xgb"  : test_ids_xgb,
        "test_ids_lgbm" : test_ids_lgbm,
    },
    FILE,
    compress=3,   # gzip‑level compression, small & fast
)
print(f"✅  Features saved → {FILE}")


✅  Features saved → /kaggle/working/polymer_feats.pkl


In [7]:
# ==== load_all_features.py ====
import joblib, os

FILE = "/kaggle/working/polymer_feats.pkl"

_loaded = joblib.load(FILE)       # returns the dict we saved
globals().update(_loaded)         # put every key back into workspace

print("✅  Loaded variables:", ", ".join(_loaded.keys()))
# now X_all_xgb, Y_all, … are ready to use


✅  Loaded variables: X_all_xgb, X_all_lgbm, idx_train, Y_all, X_test_xgb, X_test_lgbm, idx_test_xgb, idx_test_lgbm, test_ids_xgb, test_ids_lgbm


# Training 🚆

## First Class Model

### 1. XGBoost

#### *If we already know which parameter is best for model training, we only run this cell to get best model (Choose the RUN_GRID_SEARCH as False)*

In [8]:
'''# ============================================================
# 0)  global setup
# ============================================================
SAVE_DIR          = "/kaggle/working"
EARLY_STOP_ROUNDS = 100
RUN_GRID_SEARCH   = False      # True → full grid search, False → use best_params
N_FOLDS           = 5         # folds for hyper‑param search
FINAL_N_FOLDS     = 10        # NEW: folds for final model ensembling

os.makedirs(SAVE_DIR, exist_ok=True)
FIG_DIR = os.path.join(SAVE_DIR, "figs_xgb")
os.makedirs(FIG_DIR, exist_ok=True)

# X_all_xgb, Y_all, targets, X_test_xgb, test_ids_xgb, train_df, test_df
# are assumed to be already prepared earlier in the notebook/script
# --------------------------------------------------------------------

# ============================================================
# 1) hyper‑parameter grid
# ============================================================
param_grid = {
    "learning_rate":    [0.01, 0.05, 0.07],
    "max_depth":        [6, 8],
    "subsample":        [0.5, 0.6, 1.0],
    "colsample_bytree": [0.5, 0.6],
    "reg_lambda":       [0.0, 1.0, 2.0],
}

# ============================================================
# 2) manual best params  (only used when RUN_GRID_SEARCH=False)
# ============================================================
best_params = {
    "Tg":      {"tree_method": "hist", "device": "cuda", "objective": "reg:squarederror",
                "eval_metric": "mae", "learning_rate": 0.01, "max_depth": 6,
                "subsample": 0.5, "colsample_bytree": 0.5, "reg_lambda": 1.0, "seed": SEED},
    "FFV":     {"tree_method": "hist", "device": "cuda", "objective": "reg:squarederror",
                "eval_metric": "mae", "learning_rate": 0.01, "max_depth": 6,
                "subsample": 0.5, "colsample_bytree": 0.6, "reg_lambda": 2.0, "seed": SEED},
    "Tc":      {"tree_method": "hist", "device": "cuda", "objective": "reg:squarederror",
                "eval_metric": "mae", "learning_rate": 0.01, "max_depth": 6,
                "subsample": 0.6, "colsample_bytree": 0.5, "reg_lambda": 2.0, "seed": SEED},
    "Density": {"tree_method": "hist", "device": "cuda", "objective": "reg:squarederror",
                "eval_metric": "mae", "learning_rate": 0.01, "max_depth": 6,
                "subsample": 0.6, "colsample_bytree": 0.5, "reg_lambda": 0.0, "seed": SEED},
    "Rg":      {"tree_method": "hist", "device": "cuda", "objective": "reg:squarederror",
                "eval_metric": "mae", "learning_rate": 0.01, "max_depth": 8,
                "subsample": 0.6, "colsample_bytree": 0.5, "reg_lambda": 2.0, "seed": SEED},
}

# ============================================================
# 3) helper: K‑fold CV MAE
# ============================================================
cv_search = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)  # for grid search


def cv_mae(params, X, y):
    """5‑fold MAE + mean best_iteration (for grid search)"""
    ms, iters = [], []
    for tr, va in cv_search.split(X):
        booster = xgb.train(
            params,
            xgb.DMatrix(X[tr], label=y[tr]),
            num_boost_round=100_000,
            evals=[(xgb.DMatrix(X[va], label=y[va]), "val")],
            early_stopping_rounds=EARLY_STOP_ROUNDS,
            verbose_eval=False,
        )
        pred = booster.predict(
            xgb.DMatrix(X[va]), iteration_range=(0, booster.best_iteration + 1)
        )
        ms.append(mean_absolute_error(y[va], pred))
        iters.append(booster.best_iteration)
    return float(np.mean(ms)), int(np.mean(iters))

# ============================================================
# 4) main loop  (grid search + 10‑fold model ensembling)
# ============================================================
plot_data         = {p: {t: [] for t in targets} for p in param_grid}
cv_results        = {}
model_paths       = {}               # NEW: per‑target list of 10 fold models
chosen_param_set  = {}               # store full best configs
fold_mae_dict = {t: [] for t in targets}   # ← NEW

print("Running evaluation …\n")

for t_idx, tgt in enumerate(targets):
    X_tgt = X_all_xgb[np.isfinite(Y_all[:, t_idx])]
    y_tgt = Y_all[np.isfinite(Y_all[:, t_idx]), t_idx]

    # ---------- 4.1  hyper‑param search (5‑fold) ---------- #
    if RUN_GRID_SEARCH:
        best_mae, best_cfg, best_iter = 1e9, None, None
        for lr in param_grid["learning_rate"]:
            for md in param_grid["max_depth"]:
                for ss in param_grid["subsample"]:
                    for cs in param_grid["colsample_bytree"]:
                        for rl in param_grid["reg_lambda"]:
                            p = {"tree_method": "hist", "device": "cuda",
                                 "objective": "reg:squarederror",
                                 "eval_metric": "mae", "learning_rate": lr,
                                 "max_depth": md, "subsample": ss,
                                 "colsample_bytree": cs, "reg_lambda": rl,
                                 "seed": SEED}

                            mae, it_ = cv_mae(p, X_tgt, y_tgt)

                            # collect diagnostic plot points
                            plot_data["learning_rate"][tgt].append((lr, mae))
                            plot_data["max_depth"][tgt].append((md, mae))
                            plot_data["subsample"][tgt].append((ss, mae))
                            plot_data["colsample_bytree"][tgt].append((cs, mae))
                            plot_data["reg_lambda"][tgt].append((rl, mae))

                            if mae < best_mae:
                                best_mae, best_cfg, best_iter = mae, p.copy(), it_
                                
        print(f"◎ {tgt}  best grid params = {best_cfg} | "
              f"best_iteration ≈ {best_iter} | CV MAE = {best_mae:.5f}")
        chosen_params = best_cfg
    else:
        chosen_params = best_params[tgt]
        best_mae, _ = cv_mae(chosen_params, X_tgt, y_tgt)
        for pname in plot_data:
            plot_data[pname][tgt].append((chosen_params[pname], best_mae))

    chosen_param_set[tgt] = chosen_params

    # ---------- 4.2  final 10‑fold training & saving ---------- #
    cv_final = KFold(n_splits=FINAL_N_FOLDS, shuffle=True, random_state=SEED)
    fold_maes = []
    paths = []

    for f_id, (tr, va) in enumerate(cv_final.split(X_tgt)):
        booster = xgb.train(
            chosen_params,
            xgb.DMatrix(X_tgt[tr], label=y_tgt[tr]),
            num_boost_round=100_000,
            evals=[(xgb.DMatrix(X_tgt[va], label=y_tgt[va]), "val")],
            early_stopping_rounds=EARLY_STOP_ROUNDS,
            verbose_eval=False,
        )
        preds = booster.predict(
            xgb.DMatrix(X_tgt[va]), iteration_range=(0, booster.best_iteration + 1)
        )
        mae_fold = mean_absolute_error(y_tgt[va], preds)
        fold_maes.append(mae_fold)
        fold_mae_dict[tgt].append(mae_fold)

        model_path = os.path.join(SAVE_DIR, f"xgb_{tgt}_fold{f_id}.json")
        booster.save_model(model_path)
        paths.append(model_path)

    # store per‑target paths
    model_paths[tgt] = paths.copy()

    print(
        f"★ {tgt}: Fold MAEs = {[f'{m:.5f}' for m in fold_maes]} | "
        f"Mean = {np.mean(fold_maes):.5f} (best grid MAE {best_mae:.5f})"
    )

# ============================================================
# 5) 25 linear‑scale diagnostic plots (from grid search)
# ============================================================
color_map = {t: c for t, c in zip(
    targets, plt.rcParams["axes.prop_cycle"].by_key()["color"]
)}

for pname, tgt_dict in plot_data.items():          # 5 parameters
    for tgt in targets:                            # 5 targets
        pts = tgt_dict[tgt]
        if not pts:
            continue
        xs, ys = zip(*sorted(pts, key=lambda v: v[0]))
        plt.figure()
        plt.plot(xs, ys, "-o", color=color_map[tgt])
        plt.xlabel(pname)
        plt.ylabel("Mean CV MAE")
        plt.title(f"{tgt} – {pname} vs MAE")
        plt.grid(True)
        plt.savefig(os.path.join(FIG_DIR, f"{pname}_{tgt}.png"), dpi=120)
        plt.close()'''

'# ============================================================\n# 0)  global setup\n# ============================================================\nSAVE_DIR          = "/kaggle/working"\nEARLY_STOP_ROUNDS = 100\nRUN_GRID_SEARCH   = False      # True → full grid search, False → use best_params\nN_FOLDS           = 5         # folds for hyper‑param search\nFINAL_N_FOLDS     = 10        # NEW: folds for final model ensembling\n\nos.makedirs(SAVE_DIR, exist_ok=True)\nFIG_DIR = os.path.join(SAVE_DIR, "figs_xgb")\nos.makedirs(FIG_DIR, exist_ok=True)\n\n# X_all_xgb, Y_all, targets, X_test_xgb, test_ids_xgb, train_df, test_df\n# are assumed to be already prepared earlier in the notebook/script\n# --------------------------------------------------------------------\n\n# ============================================================\n# 1) hyper‑parameter grid\n# ============================================================\nparam_grid = {\n    "learning_rate":    [0.01, 0.05, 0.07],\n    "max_dept

## Calculate wMae to evaluation

In [9]:
'''# ============================================================
# compute 10-fold validation wMAE (self-check only)
# ============================================================
K = len(targets)

# 7.1  task-specific weights w_i
n_i   = {t: train_df[t].notnull().sum()                  for t in targets}
r_i   = {t: train_df[t].max() - train_df[t].min()        for t in targets}
norm  = sum(1/np.sqrt(n_i[t]) for t in targets)          # Σ_j √(1/n_j)

w_i = {
    t: (1 / r_i[t]) * (K * (1/np.sqrt(n_i[t])) / norm)
    for t in targets
}

# 7.2  fold-level weighted MAE
wmae_folds = []
for k in range(FINAL_N_FOLDS):
    wmae_k = sum(w_i[t] * fold_mae_dict[t][k] for t in targets) / K
    wmae_folds.append(wmae_k)

print("XGB: 10-fold wMAE per fold:", [f"{v:.5f}" for v in wmae_folds])
print("XGB: Mean 10-fold wMAE:", np.mean(wmae_folds))'''

'# ============================================================\n# compute 10-fold validation wMAE (self-check only)\n# ============================================================\nK = len(targets)\n\n# 7.1  task-specific weights w_i\nn_i   = {t: train_df[t].notnull().sum()                  for t in targets}\nr_i   = {t: train_df[t].max() - train_df[t].min()        for t in targets}\nnorm  = sum(1/np.sqrt(n_i[t]) for t in targets)          # Σ_j √(1/n_j)\n\nw_i = {\n    t: (1 / r_i[t]) * (K * (1/np.sqrt(n_i[t])) / norm)\n    for t in targets\n}\n\n# 7.2  fold-level weighted MAE\nwmae_folds = []\nfor k in range(FINAL_N_FOLDS):\n    wmae_k = sum(w_i[t] * fold_mae_dict[t][k] for t in targets) / K\n    wmae_folds.append(wmae_k)\n\nprint("XGB: 10-fold wMAE per fold:", [f"{v:.5f}" for v in wmae_folds])\nprint("XGB: Mean 10-fold wMAE:", np.mean(wmae_folds))'

### 2. LightGBM

In [10]:
# ============================================================
# 0)  global setup
# ============================================================
SAVE_DIR          = "/kaggle/working"
EARLY_STOP_ROUNDS = 100
RUN_GRID_SEARCH   = False      # True → full grid search, False → use best_params
N_FOLDS           = 5          # folds for hyper‑param search
FINAL_N_FOLDS     = 10         # folds for final model ensembling
SEED              = 42

os.makedirs(SAVE_DIR, exist_ok=True)
FIG_DIR = os.path.join(SAVE_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

# ============================================================
# 1) hyper‑parameter grid   (true LightGBM names)
#    ― now includes "num_leaves" so that search space matches
#      the manual best‑param templates later in the script ―
# ============================================================
param_grid = {
    "learning_rate":    [0.01, 0.03],
    "max_depth":        [6, 8],
    "num_leaves":       [63],
    "bagging_fraction": [1.0],
    "feature_fraction": [0.5],
    "lambda_l2":        [1.0],
}

# ============================================================
# 2) manual best params (one‑liner per target, true LGBM names)
#     – used when RUN_GRID_SEARCH=False –
# ============================================================
best_params = {
    "Tg":      {"objective":"regression","metric":"mae","device_type":"gpu","learning_rate":0.03,"max_depth":8,"num_leaves":63,"bagging_fraction":0.5,"bagging_freq":1,"feature_fraction":0.5,"lambda_l2":1.0,"seed":SEED,"verbosity":-1},
    "FFV":     {"objective":"regression","metric":"mae","device_type":"gpu","learning_rate":0.01,"max_depth":6,"num_leaves":63,"bagging_fraction":1.0,"bagging_freq":1,"feature_fraction":0.5,"lambda_l2":1.0,"seed":SEED,"verbosity":-1},
    "Tc":      {"objective":"regression","metric":"mae","device_type":"gpu","learning_rate":0.01,"max_depth":8,"num_leaves":63,"bagging_fraction":0.5,"bagging_freq":1,"feature_fraction":0.5,"lambda_l2":1.0,"seed":SEED,"verbosity":-1},
    "Density": {"objective":"regression","metric":"mae","device_type":"gpu","learning_rate":0.01,"max_depth":6,"num_leaves":63,"bagging_fraction":1.0,"bagging_freq":1,"feature_fraction":0.5,"lambda_l2":1.0,"seed":SEED,"verbosity":-1},
    "Rg":      {"objective":"regression","metric":"mae","device_type":"gpu","learning_rate":0.01,"max_depth":6,"num_leaves":63,"bagging_fraction":1.0,"bagging_freq":1,"feature_fraction":0.5,"lambda_l2":1.0,"seed":SEED,"verbosity":-1},
}

# ============================================================
# 3) helper: K‑fold CV MAE
# ============================================================
cv_search = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
stop_cb = [lgb.early_stopping(EARLY_STOP_ROUNDS, first_metric_only=True, verbose=False)]

def cv_mae(params, X, y):
    """N‑fold MAE + mean best_iteration (for grid search)"""
    ms, iters = [], []

    for tr, va in cv_search.split(X):
        dtrain = lgb.Dataset(X[tr], label=y[tr])
        dvalid = lgb.Dataset(X[va], label=y[va])

        booster = lgb.train(
            params,
            dtrain,
            num_boost_round=100_000,
            valid_sets=[dvalid],
            callbacks=stop_cb,          # ← here
        )
        pred = booster.predict(X[va], num_iteration=booster.best_iteration)
        ms.append(mean_absolute_error(y[va], pred))
        iters.append(booster.best_iteration)
        del dtrain, dvalid, booster
        gc.collect()
    return float(np.mean(ms)), int(np.mean(iters))

# ============================================================
# 4) main loop  (grid search + 10‑fold model ensembling)
# ============================================================
plot_data        = {p: {t: [] for t in targets} for p in param_grid}
model_paths      = {}
chosen_param_set = {}
fold_mae_dict    = {t: [] for t in targets}

print("Running LightGBM evaluation …\n")

for t_idx, tgt in enumerate(targets):
    X_tgt = X_all_lgbm[np.isfinite(Y_all[:, t_idx])]
    y_tgt = Y_all[np.isfinite(Y_all[:, t_idx]), t_idx]

    # ---------- 4.1  hyper‑param search ---------- #
    if RUN_GRID_SEARCH:
        best_mae, best_cfg, best_iter = 1e9, None, None
        for lr in param_grid["learning_rate"]:
            for md in param_grid["max_depth"]:
                for nl in param_grid["num_leaves"]:  # NEW loop level
                    for bf in param_grid["bagging_fraction"]:
                        for ff in param_grid["feature_fraction"]:
                            for l2 in param_grid["lambda_l2"]:
                                p = {
                                    "objective":        "regression",
                                    "metric":           "mae",
                                    "device_type":      "cpu",
                                    "learning_rate":    lr,
                                    "max_depth":        md,
                                    "num_leaves":       nl,  # NEW ➜ added to param dict
                                    "bagging_fraction": bf,
                                    "bagging_freq":     1,
                                    "feature_fraction": ff,
                                    "lambda_l2":        l2,
                                    "seed":             SEED,
                                    "verbosity":        -1,
                                }

                                mae, it_ = cv_mae(p, X_tgt, y_tgt)

                                # collect diagnostic plot points
                                plot_data["learning_rate"][tgt].append((lr, mae))
                                plot_data["max_depth"][tgt].append((md, mae))
                                plot_data["num_leaves"][tgt].append((nl, mae))  # NEW ➜ record
                                plot_data["bagging_fraction"][tgt].append((bf, mae))
                                plot_data["feature_fraction"][tgt].append((ff, mae))
                                plot_data["lambda_l2"][tgt].append((l2, mae))

                                if mae < best_mae:
                                    best_mae, best_cfg, best_iter = mae, p.copy(), it_
        print(f"◎ {tgt}  best grid params = {best_cfg} | "
              f"best_iteration ≈ {best_iter} | CV MAE = {best_mae:.5f}")
        chosen_params = best_cfg
    else:
        chosen_params = best_params[tgt]
        best_mae, _ = cv_mae(chosen_params, X_tgt, y_tgt)
        # record a single point for diagnostic plots, keeping grid/dataframe shape intact
        for pname in plot_data:
            plot_data[pname][tgt].append((chosen_params[pname], best_mae))

    chosen_param_set[tgt] = chosen_params

    # ---------- 4.2  final 10‑fold training & saving ---------- #
    cv_final = KFold(n_splits=FINAL_N_FOLDS, shuffle=True, random_state=SEED)
    fold_maes, paths = [], []

    for f_id, (tr, va) in enumerate(cv_final.split(X_tgt)):
        dtrain = lgb.Dataset(X_tgt[tr], label=y_tgt[tr])
        dvalid = lgb.Dataset(X_tgt[va], label=y_tgt[va])

        booster = lgb.train(
            chosen_params,
            dtrain,
            num_boost_round=100_000,
            valid_sets=[dvalid],
            callbacks=stop_cb,
        )
        preds = booster.predict(X_tgt[va], num_iteration=booster.best_iteration)
        mae_fold = mean_absolute_error(y_tgt[va], preds)
        fold_maes.append(mae_fold)
        fold_mae_dict[tgt].append(mae_fold)

        model_path = os.path.join(SAVE_DIR, f"lgb_{tgt}_fold{f_id}.txt")
        booster.save_model(model_path)
        paths.append(model_path)

        del dtrain, dvalid, booster
        gc.collect()

    model_paths[tgt] = paths.copy()

    print(
        f"★ {tgt}: Fold MAEs = {[f'{m:.5f}' for m in fold_maes]} | "
        f"Mean = {np.mean(fold_maes):.5f} (best grid MAE {best_mae:.5f})"
    )

# ============================================================
# 5) diagnostic plots  (from grid search)
# ============================================================
color_map = {t: c for t, c in zip(
    targets, plt.rcParams["axes.prop_cycle"].by_key()["color"]
)}

for pname, tgt_dict in plot_data.items():          # 6 parameters (incl. num_leaves)
    for tgt in targets:                            # 5 targets
        pts = tgt_dict[tgt]
        if not pts:
            continue
        xs, ys = zip(*sorted(pts, key=lambda v: v[0]))
        plt.figure()
        plt.plot(xs, ys, "-o", color=color_map[tgt])
        plt.xlabel(pname)
        plt.ylabel("Mean CV MAE")
        plt.title(f"{tgt} – {pname} vs MAE")
        plt.grid(True)
        plt.savefig(os.path.join(FIG_DIR, f"{pname}_{tgt}.png"), dpi=120)
        plt.close()

Running LightGBM evaluation …



1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


★ Tg: Fold MAEs = ['34.35163', '34.51671', '37.67828', '28.58988', '38.38502', '31.62146', '34.94293', '33.34444', '36.64411', '39.33901'] | Mean = 34.94135 (best grid MAE 36.15731)
★ FFV: Fold MAEs = ['0.00547', '0.00495', '0.00537', '0.00520', '0.00533', '0.00524', '0.00548', '0.00515', '0.00520', '0.00522'] | Mean = 0.00526 (best grid MAE 0.00536)
★ Tc: Fold MAEs = ['0.03159', '0.02395', '0.02648', '0.02671', '0.03450', '0.04438', '0.02626', '0.02432', '0.02198', '0.03197'] | Mean = 0.02921 (best grid MAE 0.02979)
★ Density: Fold MAEs = ['0.03546', '0.03787', '0.03735', '0.03764', '0.04075', '0.03985', '0.03917', '0.03322', '0.03101', '0.03274'] | Mean = 0.03650 (best grid MAE 0.03730)
★ Rg: Fold MAEs = ['1.50022', '1.70847', '1.57630', '1.16218', '2.13839', '1.45984', '1.27559', '1.13399', '1.50398', '1.56738'] | Mean = 1.50263 (best grid MAE 1.51379)


## Calculate wMae to evaluation

In [11]:
# ============================================================
# compute 10-fold validation wMAE (self-check only)
# ============================================================
K = len(targets)

# 7.1  task-specific weights w_i
n_i   = {t: train_df[t].notnull().sum()                  for t in targets}
r_i   = {t: train_df[t].max() - train_df[t].min()        for t in targets}
norm  = sum(1/np.sqrt(n_i[t]) for t in targets)          # Σ_j √(1/n_j)

w_i = {
    t: (1 / r_i[t]) * (K * (1/np.sqrt(n_i[t])) / norm)
    for t in targets
}

# 7.2  fold-level weighted MAE
wmae_folds = []
for k in range(FINAL_N_FOLDS):
    wmae_k = sum(w_i[t] * fold_mae_dict[t][k] for t in targets) / K
    wmae_folds.append(wmae_k)

print("LGB: 10‑fold wMAE per fold:", [f"{v:.5f}" for v in wmae_folds])
print("LGB: Mean 10‑fold wMAE:", np.mean(wmae_folds))

LGB: 10‑fold wMAE per fold: ['0.03794', '0.03947', '0.03923', '0.03206', '0.04755', '0.03935', '0.03534', '0.03200', '0.03640', '0.03969']
LGB: Mean 10‑fold wMAE: 0.03790214205398503


# Test on the test set

## XGB

In [12]:
'''print("\nBuilding submission…\n")
dtest = xgb.DMatrix(X_test_xgb, missing=np.nan)
submission = pd.DataFrame({"id": test_ids_xgb})
for tgt in targets:
    preds_ensemble = []
    for mpath in model_paths[tgt]:
        booster = xgb.Booster(); booster.load_model(mpath)
        preds_ensemble.append(booster.predict(dtest))
    submission[tgt] = np.mean(preds_ensemble, axis=0)
    print(f"{tgt} preview:\n", submission[["id", tgt]].head(3), "\n")

out_path = os.path.join(SAVE_DIR, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Submission written to {out_path}")'''

'print("\nBuilding submission…\n")\ndtest = xgb.DMatrix(X_test_xgb, missing=np.nan)\nsubmission = pd.DataFrame({"id": test_ids_xgb})\nfor tgt in targets:\n    preds_ensemble = []\n    for mpath in model_paths[tgt]:\n        booster = xgb.Booster(); booster.load_model(mpath)\n        preds_ensemble.append(booster.predict(dtest))\n    submission[tgt] = np.mean(preds_ensemble, axis=0)\n    print(f"{tgt} preview:\n", submission[["id", tgt]].head(3), "\n")\n\nout_path = os.path.join(SAVE_DIR, "submission.csv")\nsubmission.to_csv(out_path, index=False)\nprint(f"Submission written to {out_path}")'

## LGB

In [13]:
print("\nBuilding submission …\n")
submission = pd.DataFrame({"id": test_ids_lgbm})
for tgt in targets:
    preds_ens = []
    for pth in model_paths[tgt]:
        booster = lgb.Booster(model_file=pth)
        preds_ens.append(
            booster.predict(X_test_lgbm, num_iteration=booster.best_iteration)
        )
    submission[tgt] = np.mean(preds_ens, axis=0)
    print(f"{tgt} preview:\n", submission[['id', tgt]].head(3), "\n")

out_path = os.path.join(SAVE_DIR, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"✅  Submission written to {out_path}")



Building submission …

Tg preview:
            id          Tg
0  1109053969  179.054584
1  1422188626  199.762870
2  2032016830  134.102281 

FFV preview:
            id       FFV
0  1109053969  0.374977
1  1422188626  0.375723
2  2032016830  0.350714 

Tc preview:
            id        Tc
0  1109053969  0.188075
1  1422188626  0.239370
2  2032016830  0.243057 

Density preview:
            id   Density
0  1109053969  1.208565
1  1422188626  1.087974
2  2032016830  1.084361 

Rg preview:
            id         Rg
0  1109053969  22.000533
1  1422188626  19.882091
2  2032016830  21.133740 

✅  Submission written to /kaggle/working/submission.csv
